# TUẦN 3 - PHẦN 2: LƯU TRỮ PHÂN TÁN
**Sinh viên:** Hoàng Ngọc Dung  
**MSSV:** 23139006

In [ ]:
import os
import time
import numpy as np
import pandas as pd


## Bài 1 - So sánh CSV và Parquet với dữ liệu 20 cột

In [ ]:
np.random.seed(42)
n = 200_000

df = pd.DataFrame({
    'id': np.arange(n),
    'khu_vuc': np.random.choice(['HCM', 'HN', 'DN', 'CT', 'HP'], n),
    'gia_tri': np.random.exponential(50_000, n).round(2),
    'so_luong': np.random.randint(1, 20, n),
    'ngay': pd.to_datetime('2026-01-01') + pd.to_timedelta(np.random.randint(0, 240, n), unit='D'),
    'ma_khach_hang': np.random.randint(10000, 99999, n),
    'ma_san_pham': np.random.randint(1000, 9999, n),
    'kenh_ban': np.random.choice(['Online', 'Store', 'App'], n),
    'phuong_thuc_tt': np.random.choice(['Cash', 'Card', 'EWallet', 'Bank'], n),
    'trang_thai': np.random.choice(['Completed', 'Pending', 'Cancelled'], n),
    'giam_gia': np.random.uniform(0, 0.3, n).round(3),
    'phi_van_chuyen': np.random.uniform(0, 50_000, n).round(2),
    'thue': np.random.uniform(0, 0.1, n).round(3),
    'diem_danh_gia': np.random.randint(1, 6, n),
    'nhom_khach_hang': np.random.choice(['New', 'Regular', 'VIP'], n),
    'thanh_pho': np.random.choice(['HCM', 'Ha Noi', 'Da Nang', 'Can Tho', 'Hai Phong'], n),
    'kho': np.random.choice(['K1', 'K2', 'K3', 'K4'], n),
    'nhan_vien': np.random.randint(1, 501, n),
    'loai_don': np.random.choice(['Retail', 'Wholesale', 'Return'], n),
    'uu_tien': np.random.choice(['Low', 'Normal', 'High'], n)
})

df.head()

In [ ]:
csv_file = 'bench_20cols.csv'
parquet_file = 'bench_20cols.parquet'

df.to_csv(csv_file, index=False)
df.to_parquet(parquet_file, engine='pyarrow', compression='snappy', index=False)

csv_size = os.path.getsize(csv_file) / 1e6
parquet_size = os.path.getsize(parquet_file) / 1e6

columns_to_read = ['gia_tri', 'so_luong']

t0 = time.perf_counter()
pd.read_csv(csv_file, usecols=columns_to_read)
t_csv = time.perf_counter() - t0

t0 = time.perf_counter()
pd.read_parquet(parquet_file, columns=columns_to_read)
t_parquet = time.perf_counter() - t0

print(f'Dung lượng CSV     : {csv_size:.2f} MB')
print(f'Dung lượng Parquet : {parquet_size:.2f} MB')
print(f'Parquet/CSV        : {parquet_size / csv_size * 100:.2f}%')
print(f'Thời gian đọc CSV  : {t_csv:.4f} s')
print(f'Thời gian đọc Parquet: {t_parquet:.4f} s')
print(f'CSV/Parquet        : {t_csv / t_parquet:.2f} lần')

**Nhận xét:** Với 20 cột nhưng chỉ đọc 2 cột, Parquet có lợi thế rõ hơn nhờ lưu trữ theo cột và column pruning. Chỉ 2/20 = 10% số cột được yêu cầu, nên mức tiết kiệm I/O lý tưởng theo số cột có thể đạt khoảng 10 lần so với đọc đủ 20 cột. Kết quả dung lượng và thời gian thực tế phụ thuộc máy chạy.

## Bài 2 - Khởi động Iceberg + Spark bằng Docker
1. Cài Docker Desktop hoặc Docker Engine có Docker Compose.
2. Mở terminal tại thư mục `iceberg-spark-docker-quickstart`.
3. Chạy `docker compose up` hoặc `docker compose up -d`.
4. Kiểm tra bằng `docker compose ps`.
5. Mở Spark SQL bằng `docker exec -it spark-iceberg spark-sql`, hoặc PySpark bằng `docker exec -it spark-iceberg pyspark`.
6. Kết thúc bằng `docker compose down`.

## Bài 3 - Tính lại Lab 1 với 2 PB và replication factor = 2

In [1]:
data_pb = 2.0
replication_factor = 2

physical_replication = data_pb * replication_factor
hot_data = data_pb * 0.20
cold_data = data_pb * 0.80
hot_physical = hot_data * replication_factor
cold_physical = cold_data * 1.5
hybrid_total = hot_physical + cold_physical
saving_percent = (physical_replication - hybrid_total) / physical_replication * 100

print(f'Toàn bộ dùng replication: {data_pb:.1f} PB x {replication_factor} = {physical_replication:.1f} PB')
print(f'20% dữ liệu nóng: {hot_data:.1f} PB x 2 = {hot_physical:.1f} PB')
print(f'80% dữ liệu lạnh: {cold_data:.1f} PB x 1.5 = {cold_physical:.1f} PB')
print(f'Tổng dung lượng mới: {hybrid_total:.1f} PB')
print(f'Tiết kiệm so với replication toàn bộ: {saving_percent:.1f}%')

Toàn bộ dùng replication: 2.0 PB x 2 = 4.0 PB
20% dữ liệu nóng: 0.4 PB x 2 = 0.8 PB
80% dữ liệu lạnh: 1.6 PB x 1.5 = 2.4 PB
Tổng dung lượng mới: 3.2 PB
Tiết kiệm so với replication toàn bộ: 20.0%


**Nhận xét:** RF = 2 tiết kiệm dung lượng hơn RF = 3 nhưng khả năng chịu lỗi thấp hơn vì mỗi block chỉ còn hai bản sao. Erasure Coding phù hợp với dữ liệu lạnh vì giảm chi phí lưu trữ, đổi lại quá trình phục hồi phức tạp hơn replication.